# ROI (Region-of-Interest) analysis and retraining

ROI (Region-of-Interest) analysis and retraining pipeline
Addresses the reviewer's still-open comment:
  "we suggest re-training with an explicit udder region-of-interest crop
   and reporting the percentage of test images whose peak activation
   falls outside of a segmented udder mask."

This script has TWO independent parts. You can run Part A alone first
(it does NOT require retraining anything, just your already-trained
DenseNet-121 checkpoint) and only run Part B if you have time/compute
for a full retraining pass.

  PART A: quantitative Grad-CAM vs. auto-detected udder mask analysis
          -> answers "what % of peak activation falls outside the udder mask"
  PART B: crop every image to its detected udder ROI and retrain
          DenseNet-121 from scratch on the cropped images, then compare
          to your original (uncropped) results

HOW TO USE
  1. Open this file in Colab (File > Upload notebook, or paste cells into
     a fresh notebook -- the "# %%" markers below are cell breaks).
  2. Edit the CONFIG block to match your paths and checkpoint filename.
  3. Run the "sanity check" cell in Part A FIRST and LOOK at the printed
     mask overlays before trusting any downstream number. The automatic
     mask (Otsu threshold + largest warm blob) is a heuristic, not a
     verified segmentation -- if it looks wrong on your images, the
     thresholding parameters need adjusting before the rest of the
     pipeline is meaningful.

WHAT THE AUTOMATIC MASK ACTUALLY IS (be upfront about this in the paper)
  There is no manual udder segmentation for this archive, so the "udder
  mask" here is derived automatically: convert to grayscale, Otsu-threshold
  to separate warm (bright) tissue from the cooler background, keep the
  single largest connected bright component, then clean it up with
  morphological opening/closing. This works well when the udder is the
  dominant warm object in frame (which the acquisition protocol -- close-up
  udder shots -- makes likely) but can fail on atypical frames (e.g. a leg
  or a warm floor patch in view). Report it in the manuscript exactly as
  what it is: an automatic, intensity-based proxy for the udder region,
  not a verified anatomical segmentation -- and say so in Methods/Limitations.

## CELL 1 - CONFIG (edit this block to match your environment)

In [ ]:
import os

# --- paths: adjust to match your Drive layout ---
ROOT = "/content/drive/MyDrive/mastitis_dataset"
OUT_DIR = os.path.join(ROOT, "DL_results")
ROI_OUT_DIR = os.path.join(ROOT, "DL_results_ROI")   # new outputs go here, originals untouched
os.makedirs(ROI_OUT_DIR, exist_ok=True)

# --- image folders: adjust if your class-folder names differ ---
# Assumed layout, matching the companion-paper archive:
#   ROOT/Healthy/*.bmp
#   ROOT/Mastitis/*.bmp   (rename to whatever your two class folders are actually called)
CLASS_FOLDERS = {
    "Healthy": 0,
    "Mastitis": 1,
}

# --- checkpoint of the model you already trained and are reporting on ---
BEST_MODEL_TIMM_NAME = "densenet121"          # matches MODEL_ZOO["DenseNet-121"]
BEST_MODEL_CKPT = os.path.join(OUT_DIR, f"{BEST_MODEL_TIMM_NAME}.pt")

# --- reproducibility: MUST match the RANDOM_STATE / seed used in your
#     original DL notebook so the held-out split is identical and the
#     ROI-vs-original comparison is apples-to-apples ---
RANDOM_STATE = 42   # <-- CHANGE THIS to whatever you actually used

IMG_SIZE = 224

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## CELL 2 - collect image paths + labels

In [ ]:
def collect_images(root, class_folders):
    rows = []
    for folder_name, label in class_folders.items():
        folder_path = os.path.join(root, folder_name)
        if not os.path.isdir(folder_path):
            print(f"WARNING: folder not found: {folder_path} -- fix CLASS_FOLDERS in CONFIG")
            continue
        for fname in sorted(os.listdir(folder_path)):
            if fname.lower().endswith((".bmp", ".png", ".jpg", ".jpeg")):
                rows.append({"path": os.path.join(folder_path, fname), "label": label, "fname": fname})
    df = pd.DataFrame(rows)
    print(f"Collected {len(df)} images ({(df.label==0).sum()} healthy, {(df.label==1).sum()} mastitic)")
    return df

all_images_df = collect_images(ROOT, CLASS_FOLDERS)

## CELL 3 - automatic udder ROI mask (Otsu + largest warm blob)

In [ ]:
def load_bgr(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)  # BGR
    if img is None:
        # fall back to PIL for odd BMP variants OpenCV chokes on
        pil = Image.open(path).convert("RGB")
        img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    return img


def derive_udder_mask(bgr_img, margin_frac=0.08, min_area_frac=0.03):
    """
    Returns (mask, bbox) where mask is a uint8 0/255 image same size as
    input, and bbox is (x1, y1, x2, y2) of the mask's bounding box WITH
    margin added and clipped to image bounds. bbox is None if no
    sufficiently large warm blob was found (caller should skip / flag
    these images rather than silently cropping garbage).
    """
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu threshold on brightness: udder region assumed warmer/brighter
    # than surrounding background in these rendered thermal palettes.
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)

    # keep largest connected component only
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(th, connectivity=8)
    if n_labels <= 1:
        return None, None

    # stats[0] is background; find largest non-background component
    areas = stats[1:, cv2.CC_STAT_AREA]
    if len(areas) == 0:
        return None, None
    largest_idx = 1 + int(np.argmax(areas))
    area = stats[largest_idx, cv2.CC_STAT_AREA]

    h, w = gray.shape
    if area < min_area_frac * h * w:
        return None, None  # too small to trust as "the udder"

    mask = np.where(labels == largest_idx, 255, 0).astype(np.uint8)

    x, y, bw, bh = (stats[largest_idx, cv2.CC_STAT_LEFT], stats[largest_idx, cv2.CC_STAT_TOP],
                     stats[largest_idx, cv2.CC_STAT_WIDTH], stats[largest_idx, cv2.CC_STAT_HEIGHT])
    mx, my = int(bw * margin_frac), int(bh * margin_frac)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)

    return mask, (x1, y1, x2, y2)


# --- SANITY CHECK: run this before anything else and LOOK at the output ---
def sanity_check_masks(df, n=8, seed=0):
    sample = df.sample(n=min(n, len(df)), random_state=seed)
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    for i, (_, row) in enumerate(sample.iterrows()):
        bgr = load_bgr(row["path"])
        mask, bbox = derive_udder_mask(bgr)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(rgb)
        axes[0, i].set_title(f"{row['fname'][:14]}\n({'H' if row['label']==0 else 'M'})", fontsize=8)
        axes[0, i].axis("off")
        overlay = rgb.copy()
        if mask is not None:
            overlay[mask > 0] = (0.5 * overlay[mask > 0] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
            x1, y1, x2, y2 = bbox
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 3)
        axes[1, i].imshow(overlay)
        axes[1, i].set_title("mask (red) + crop bbox (green)" if mask is not None else "NO MASK FOUND", fontsize=8)
        axes[1, i].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(ROI_OUT_DIR, "mask_sanity_check.png"), dpi=120)
    plt.show()
    print(f"Saved sanity-check grid to {os.path.join(ROI_OUT_DIR, 'mask_sanity_check.png')}")
    print("LOOK AT THIS IMAGE before trusting anything downstream. If the green boxes")
    print("are not tightly around the udder on most images, tighten/loosen")
    print("min_area_frac / margin_frac in derive_udder_mask(), or the Otsu")
    print("approach is not suitable for this archive and a different heuristic")
    print("(e.g. fixed center-crop, since acquisition is close-up and framed)")
    print("should be used and documented instead.")

sanity_check_masks(all_images_df, n=8)

## CELL 4 - load the trained model + Grad-CAM (auto-detects last conv layer)

In [ ]:
def load_trained_model(timm_name, ckpt_path, num_classes=2):
    model = timm.create_model(timm_name, pretrained=False, num_classes=num_classes)
    state = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model


def find_last_conv_layer(model):
    last_conv = None
    last_name = None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
            last_name = name
    if last_conv is None:
        raise RuntimeError("No Conv2d layer found in model -- check architecture.")
    print(f"Grad-CAM target layer auto-detected: {last_name}")
    return last_conv


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        score = output[:, class_idx]
        score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # GAP over spatial dims
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam[0, 0].cpu().numpy()
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam, class_idx, torch.softmax(output, dim=1)[0].detach().cpu().numpy()


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

def preprocess_for_model(bgr_img, size=IMG_SIZE):
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (size, size))
    x = rgb.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    x = torch.tensor(x.transpose(2, 0, 1), dtype=torch.float32).unsqueeze(0)
    return x.to(DEVICE)


best_model = load_trained_model(BEST_MODEL_TIMM_NAME, BEST_MODEL_CKPT)
target_layer = find_last_conv_layer(best_model)
gradcam = GradCAM(best_model, target_layer)

## CELL 5 - PART A: quantify % of peak Grad-CAM activation outside the udder mask

In [ ]:
def peak_outside_mask_fraction(cam, mask, size, top_frac=0.20):
    """
    cam: 2D array (any size) of Grad-CAM values in [0, 1]
    mask: 2D uint8 0/255 array, ORIGINAL image resolution
    size: (h, w) of the original image, to resize cam onto the same grid as mask
    top_frac: fraction of pixels (by CAM value) counted as "peak activation"
    Returns fraction of peak-activation pixels that fall OUTSIDE the mask.
    """
    h, w = size
    cam_resized = cv2.resize(cam, (w, h))
    flat = cam_resized.flatten()
    n_top = max(1, int(top_frac * flat.size))
    thresh = np.partition(flat, -n_top)[-n_top]
    peak_binary = (cam_resized >= thresh)

    mask_binary = (mask > 0)
    outside = peak_binary & (~mask_binary)
    frac_outside = outside.sum() / max(1, peak_binary.sum())
    return frac_outside


def run_saliency_vs_mask_analysis(df, model, gradcam_obj, top_frac=0.20):
    rows = []
    skipped = 0
    for _, row in df.iterrows():
        bgr = load_bgr(row["path"])
        h, w = bgr.shape[:2]
        mask, bbox = derive_udder_mask(bgr)
        if mask is None:
            skipped += 1
            continue

        x = preprocess_for_model(bgr)
        cam, pred_class, probs = gradcam_obj(x, class_idx=None)  # explain the model's own top prediction
        frac_outside = peak_outside_mask_fraction(cam, mask, (h, w), top_frac=top_frac)

        rows.append({
            "fname": row["fname"],
            "true_label": row["label"],
            "pred_label": pred_class,
            "correct": int(row["label"] == pred_class),
            "prob_mastitic": float(probs[1]),
            "frac_peak_activation_outside_udder_mask": frac_outside,
        })

    result_df = pd.DataFrame(rows)
    print(f"Analyzed {len(result_df)} images, skipped {skipped} (no reliable mask found)")
    return result_df


# NOTE: run this on whichever split you intend to report (held-out test set
# recommended, to match Section 3.1/3.4 where Grad-CAM examples are shown).
# If you saved the held-out test index list from the original notebook,
# filter all_images_df down to it before calling this. Otherwise this runs
# on the full 976-image archive, which you should say explicitly if you
# report it that way.
saliency_df = run_saliency_vs_mask_analysis(all_images_df, best_model, gradcam, top_frac=0.20)
saliency_df.to_csv(os.path.join(ROI_OUT_DIR, "saliency_vs_udder_mask.csv"), index=False)

summary = {
    "n_images": len(saliency_df),
    "mean_frac_outside": saliency_df["frac_peak_activation_outside_udder_mask"].mean(),
    "median_frac_outside": saliency_df["frac_peak_activation_outside_udder_mask"].median(),
    "mean_frac_outside_correct": saliency_df.loc[saliency_df.correct == 1, "frac_peak_activation_outside_udder_mask"].mean(),
    "mean_frac_outside_incorrect": saliency_df.loc[saliency_df.correct == 0, "frac_peak_activation_outside_udder_mask"].mean(),
}
print(summary)
pd.Series(summary).to_csv(os.path.join(ROI_OUT_DIR, "saliency_vs_udder_mask_SUMMARY.csv"))

print("\nDrop these numbers into the manuscript once you have them, e.g.:")
print('  "Across the held-out test set, a mean of {:.1f}% (median {:.1f}%) of each '
      'image\'s peak Grad-CAM activation (top 20% of activation values) fell outside '
      'an automatically detected udder region (Otsu-thresholded largest warm blob, '
      'with manual spot-checking; Supplementary Figure SX)."'.format(
          summary["mean_frac_outside"] * 100, summary["median_frac_outside"] * 100))

## CELL 6 - PART B (optional, heavier): crop to ROI and retrain DenseNet-121

In [ ]:
# Only run this if Part A's sanity check looked good AND you have GPU time
# for a full retrain (same protocol as your original run: 20 epochs, AdamW,
# lr=3e-4, weight_decay=1e-4, same held-out split).

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

def build_roi_cropped_dataset(df, out_dir, size=IMG_SIZE):
    """
    Crops every image to its auto-detected udder ROI and saves the cropped,
    resized copy to out_dir. Images with no reliable mask are copied
    uncropped (center-cropped as a fallback) and flagged in the returned
    dataframe so you can decide whether to exclude them.
    """
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    for _, row in df.iterrows():
        bgr = load_bgr(row["path"])
        h, w = bgr.shape[:2]
        mask, bbox = derive_udder_mask(bgr)
        if bbox is not None:
            x1, y1, x2, y2 = bbox
            crop = bgr[y1:y2, x1:x2]
            fallback = False
        else:
            # fallback: center crop at 70% scale, matching typical framing,
            # rather than silently including an unrepresentative full frame
            cx, cy = w // 2, h // 2
            half_w, half_h = int(w * 0.35), int(h * 0.35)
            crop = bgr[max(0, cy - half_h):cy + half_h, max(0, cx - half_w):cx + half_w]
            fallback = True

        crop = cv2.resize(crop, (size, size))
        out_path = os.path.join(out_dir, row["fname"])
        cv2.imwrite(out_path, crop)
        rows.append({**row.to_dict(), "roi_path": out_path, "used_fallback_crop": fallback})

    out_df = pd.DataFrame(rows)
    n_fallback = out_df["used_fallback_crop"].sum()
    print(f"Built ROI-cropped dataset: {len(out_df)} images, {n_fallback} used the fallback center-crop")
    return out_df


roi_df = build_roi_cropped_dataset(all_images_df, os.path.join(ROI_OUT_DIR, "cropped_images"))
roi_df.to_csv(os.path.join(ROI_OUT_DIR, "roi_dataset_manifest.csv"), index=False)


class ROIImageDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        bgr = cv2.imread(row["roi_path"], cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        if self.augment:
            if np.random.rand() < 0.5:
                rgb = np.fliplr(rgb).copy()
        x = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        x = torch.tensor(x.transpose(2, 0, 1), dtype=torch.float32)
        y = torch.tensor(int(row["label"]), dtype=torch.long)
        return x, y


def train_densenet_on_roi(roi_df, random_state=RANDOM_STATE, epochs=20, lr=3e-4, weight_decay=1e-4, batch_size=32):
    """
    Same optimizer/schedule as the original run. IMPORTANT: for a fair
    comparison, this should really use the *same* group-aware (cow-level
    identifier) split as your original held-out evaluation, not a fresh
    random split. If you saved the original split's filenames, load them
    here and filter roi_df to match instead of calling train_test_split.
    The stratified split below is a fallback if you did not save that.
    """
    train_df, test_df = train_test_split(
        roi_df, test_size=0.15, stratify=roi_df["label"], random_state=random_state
    )
    train_df, val_df = train_test_split(
        train_df, test_size=0.15, stratify=train_df["label"], random_state=random_state
    )
    print(f"ROI split: train={len(train_df)} val={len(val_df)} test={len(test_df)}")

    train_loader = DataLoader(ROIImageDataset(train_df, augment=True), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(ROIImageDataset(val_df, augment=False), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(ROIImageDataset(test_df, augment=False), batch_size=batch_size, shuffle=False)

    model = timm.create_model(BEST_MODEL_TIMM_NAME, pretrained=True, num_classes=2).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_val_auc = -1
    best_state = None
    from sklearn.metrics import roc_auc_score

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        model.eval()
        val_probs, val_true = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE)
                out = torch.softmax(model(x), dim=1)[:, 1].cpu().numpy()
                val_probs.extend(out.tolist())
                val_true.extend(y.numpy().tolist())
        val_auc = roc_auc_score(val_true, val_probs)
        print(f"epoch {epoch+1}/{epochs}  val_AUC={val_auc:.4f}")
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(ROI_OUT_DIR, f"{BEST_MODEL_TIMM_NAME}_ROI.pt"))

    # final held-out test evaluation
    model.eval()
    test_probs, test_true = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            out = torch.softmax(model(x), dim=1)[:, 1].cpu().numpy()
            test_probs.extend(out.tolist())
            test_true.extend(y.numpy().tolist())

    test_probs = np.array(test_probs)
    test_true = np.array(test_true)
    test_pred = (test_probs >= 0.5).astype(int)

    from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                                  recall_score, f1_score, confusion_matrix)
    tn, fp, fn, tp = confusion_matrix(test_true, test_pred).ravel()
    metrics = {
        "accuracy": accuracy_score(test_true, test_pred),
        "auc": roc_auc_score(test_true, test_probs),
        "sensitivity": recall_score(test_true, test_pred),
        "specificity": tn / (tn + fp),
        "precision_ppv": precision_score(test_true, test_pred),
        "npv": tn / (tn + fn) if (tn + fn) > 0 else float("nan"),
        "f1": f1_score(test_true, test_pred),
        "n_test": len(test_true),
    }
    print("ROI-cropped DenseNet-121 held-out test metrics:", metrics)
    pd.Series(metrics).to_csv(os.path.join(ROI_OUT_DIR, "densenet121_ROI_test_metrics.csv"))
    return metrics, model


# Uncomment to actually run the retrain (GPU recommended; ~20 epochs on
# ~976 small crops should be a few minutes on a Colab T4):
# roi_metrics, roi_model = train_densenet_on_roi(roi_df)
# print("\nCompare directly to your original Table 3 DenseNet-121 row",
#       "(accuracy 0.822, AUC 0.922, sensitivity 0.900, specificity 0.792).")